# PC03 · DrawCAPTCHA: verificación humana mediante dibujo y CNN

Curso: Procesamiento de Imágenes | Alumno: [completar] | Código: [completar]

## 1. Planteamiento del problema
Los programas de OCR y las redes neuronales ya resuelven muchos CAPTCHA de texto distorsionado, y los de selección de imágenes son molestos y poco accesibles en celular. Proponemos DrawCAPTCHA: el sistema pide dibujar una figura elegida al azar y una red neuronal convolucional verifica que el dibujo corresponde a lo pedido. Si coincide con suficiente confianza concede el acceso; si no, lo deniega.

Usamos 5 categorías: estrella, rayo, espiral, corazón y nube.

## 2. Justificación
Un bot tiene que generar un trazo que la red acepte para una figura aleatoria, lo que es más costoso que leer texto. Dibujar con el dedo también es natural en celulares y tablets. Además de la exactitud medimos FAR (tasa de falsa aceptación) y FRR (tasa de falso rechazo), igual que en los sistemas biométricos.

## 3. Mejoras aplicadas a la red
La consigna pide al menos 3; aplicamos las 5.

| # | Mejora | Cómo se aplica |
|---|--------|----------------|
| 1 | Tamaño de entrada | 28×28 (base) → 64×64 |
| 2 | Más épocas | 10 (base) → hasta 80 con *EarlyStopping* |
| 3 | Modificar capas | BatchNorm, más filtros (32→256), GlobalAveragePooling, *data augmentation* |
| 4 | Dropout | 0.2, 0.3, 0.5 y 0.3 en distintas profundidades |
| 5 | Skip connections | Bloques residuales tipo ResNet |

## 4. Configuración e importaciones
Conviene ejecutarlo en Google Colab con GPU (*Entorno de ejecución → Cambiar tipo → T4 GPU*).

In [ ]:
import os, glob, zipfile, random, base64, io, itertools, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
import tensorflow as tf
import keras
from keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report, accuracy_score

CLASES = ['estrella', 'rayo', 'espiral', 'corazon', 'nube']
SIZE = 64          # la red base usa 28x28
EPOCAS = int(os.environ.get('PC03_EPOCAS', 80))   # máximo; EarlyStopping detiene si deja de mejorar
SEED = 42
SPLIT_POR_AUTOR = True    # True: prueba con personas que la red nunca vio, más realista; requiere 3+ autores
FAR_MAX = 0.01     # FAR máximo aceptado en validación al elegir el umbral del CAPTCHA
N_GARABATOS = 500  # intentos simulados de un bot que traza líneas al azar

random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print('TensorFlow', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))

## 5. Carga del dataset
Capturamos el dataset con nuestra página web (`captura/index.html`). La carpeta `dataset` se comprime como `dataset.zip` y se sube cuando la celda lo pide.

In [ ]:
def buscar_dataset():
    for raiz, dirs, _ in os.walk('.'):
        if all(c in dirs for c in CLASES):
            return raiz
    return None

RUTA = os.environ.get('PC03_DATASET') or buscar_dataset()   # PC03_DATASET: solo para pruebas de humo
if RUTA is None:
    try:
        from google.colab import files  # type: ignore
        print('Subir dataset.zip ...')
        subido = files.upload()
        for nombre in subido:
            if nombre.endswith('.zip'):
                zipfile.ZipFile(nombre).extractall('datos')
    except ImportError:
        print('Fuera de Colab: la carpeta dataset debe estar junto al notebook.')
    RUTA = buscar_dataset()
if RUTA is None:
    raise FileNotFoundError('No se encontró el dataset: faltan las carpetas de las 5 clases.')

print('Dataset encontrado en:', RUTA)
if 'sintetico' in RUTA.lower():
    print('⚠️ DATOS SINTÉTICOS: solo prueban que el notebook corre. Estas métricas no se reportan.')
for c in CLASES:
    print(f'  {c:<9}', len(glob.glob(os.path.join(RUTA, c, '*.png'))), 'imágenes')

### 5.1 Preprocesamiento
1. Escala de grises e inversión (trazo blanco sobre fondo negro, como MNIST).
2. Recorte al contenido (*bounding box*) y centrado en un cuadrado con margen, para que la red no dependa de dónde se dibujó.
3. Redimensionado a `SIZE×SIZE` y normalización a [0, 1].

In [ ]:
def preprocesar(img, size=SIZE):
    g = 255.0 - np.asarray(img.convert('L'), dtype=np.float32)
    ys, xs = np.where(g > 40)
    if len(xs) > 0:
        g = g[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
    h, w = g.shape
    lado = int(max(h, w) * 1.2) + 2                       # cuadrado con 10% de margen
    lienzo = np.zeros((lado, lado), dtype=np.float32)
    y0, x0 = (lado - h) // 2, (lado - w) // 2
    lienzo[y0:y0 + h, x0:x0 + w] = g
    im = Image.fromarray(lienzo.astype(np.uint8)).resize((size, size), Image.LANCZOS)
    return (np.asarray(im, dtype=np.float32) / 255.0)[..., None]

X, y, autores = [], [], []
for i, c in enumerate(CLASES):
    for f in sorted(glob.glob(os.path.join(RUTA, c, '*.png'))):
        X.append(preprocesar(Image.open(f)))
        y.append(i)
        partes = os.path.basename(f).split('_')
        autores.append(partes[1] if len(partes) >= 3 else 'desconocido')
X, y, autores = np.array(X), np.array(y), np.array(autores)
print('X:', X.shape, '| y:', y.shape, '| autores:', sorted(set(autores)))

### 5.2 Exploración visual del dataset

In [ ]:
fig, ejes = plt.subplots(len(CLASES), 8, figsize=(12, 7.5))
for i, c in enumerate(CLASES):
    idx = np.random.choice(np.where(y == i)[0], 8, replace=False)
    for j, k in enumerate(idx):
        ejes[i, j].imshow(X[k, ..., 0], cmap='gray'); ejes[i, j].axis('off')
    ejes[i, 0].set_title(c, loc='left', fontsize=11, fontweight='bold')
plt.suptitle(f'Muestras preprocesadas ({SIZE}x{SIZE})'); plt.tight_layout(); plt.show()

fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
pd.Series(y).map(dict(enumerate(CLASES))).value_counts().reindex(CLASES).plot.bar(ax=ax[0], color='#2f6fed')
ax[0].set_title('Imágenes por clase'); ax[0].tick_params(axis='x', rotation=0)
pd.crosstab(autores, pd.Series(y).map(dict(enumerate(CLASES))).values).reindex(columns=CLASES).plot.bar(stacked=True, ax=ax[1])
ax[1].set_title('Imágenes por autor'); ax[1].tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.show()

### 5.3 División entrenamiento / validación / prueba (70 / 15 / 15)
Con `SPLIT_POR_AUTOR = True` cada autor queda completo en un solo conjunto, así que validación y prueba usan personas que la red nunca vio, como en un CAPTCHA real. Se elige el reparto de autores más cercano a 70/15/15 en imágenes; como los autores no se parten, el porcentaje puede ser aproximado y la celda imprime el real.

Con menos de 3 autores no se puede dividir así: la celda avisa y usa una división estratificada por clase, exacta en 70/15/15, cuyas métricas no prueban generalización a personas nuevas.

Hay que revisar la lista de autores que imprime la sección 5.1: una misma persona escrita con dos nombres cuenta como dos autores y filtra datos entre conjuntos.

In [ ]:
def dividir_por_autor(autores, y, objetivo=(0.70, 0.15, 0.15)):
    """Reparte autores completos lo más cerca posible de `objetivo` en imágenes, o devuelve None."""
    # Límite: prueba 3^n repartos, rápido hasta unos 12 autores; con más haría falta una búsqueda voraz
    nombres = np.random.default_rng(SEED).permutation(sorted(set(autores)))
    conteo = np.array([np.bincount(y[autores == a], minlength=len(CLASES)) for a in nombres])
    mejor, menor_error = None, np.inf
    for reparto in itertools.product(range(3), repeat=len(nombres)):
        reparto = np.array(reparto)
        por_conjunto = [conteo[reparto == s].sum(axis=0) for s in range(3)]
        if any(c.min() == 0 for c in por_conjunto):   # cada conjunto necesita las 5 clases
            continue
        error = sum(abs(c.sum() / len(y) - o) for c, o in zip(por_conjunto, objetivo))
        if error < menor_error:
            mejor, menor_error = reparto, error
    if mejor is None:
        return None
    return [np.where(np.isin(autores, nombres[mejor == s]))[0] for s in range(3)]

idx = np.arange(len(y))
conjuntos = dividir_por_autor(autores, y) if SPLIT_POR_AUTOR else None
if conjuntos is not None:
    idx_tr, idx_va, idx_te = conjuntos
    MODO_SPLIT = 'por autor'
    assert not (set(autores[idx_tr]) & set(autores[idx_va]) or set(autores[idx_tr]) & set(autores[idx_te])
                or set(autores[idx_va]) & set(autores[idx_te])), 'un autor quedó en dos conjuntos'
else:
    if SPLIT_POR_AUTOR:
        print('⚠️ NO SE PUDO DIVIDIR POR AUTOR: se necesitan al menos 3 autores con las 5 clases.')
        print('⚠️ Se usa división estratificada: estas métricas no prueban generalización a personas nuevas.')
    idx_tr, idx_tmp = train_test_split(idx, test_size=0.30, stratify=y, random_state=SEED)
    idx_va, idx_te = train_test_split(idx_tmp, test_size=0.50, stratify=y[idx_tmp], random_state=SEED)
    MODO_SPLIT = 'estratificado'

assert np.array_equal(np.sort(np.concatenate([idx_tr, idx_va, idx_te])), idx), 'la división perdió o repitió imágenes'
assert all(len(set(y[i])) == len(CLASES) for i in (idx_tr, idx_va, idx_te)), 'falta una clase en algún conjunto'

X_tr, y_tr = X[idx_tr], y[idx_tr]
X_va, y_va = X[idx_va], y[idx_va]
X_te, y_te = X[idx_te], y[idx_te]
print(f'Modo: {MODO_SPLIT}')
for nombre, i in [('Entrenamiento', idx_tr), ('Validación', idx_va), ('Prueba', idx_te)]:
    print(f'{nombre}: {len(i)} imágenes ({len(i) / len(y):.1%}) | {len(set(autores[i]))} autores')

## 6. Modelo base (28×28, 10 épocas)
Es una CNN simple como la vista en clase. La usamos como referencia para medir cuánto aportan las mejoras.

In [ ]:
def a28(A):
    return tf.image.resize(A, (28, 28)).numpy()
X_tr28, X_va28, X_te28 = a28(X_tr), a28(X_va), a28(X_te)

def crear_cnn_simple(size, dropout=False):
    capas = [layers.Input((size, size, 1))]
    for filtros, tasa in [(32, 0.2), (64, 0.3)]:
        capas += [layers.Conv2D(filtros, 3, activation='relu'), layers.MaxPooling2D()]
        if dropout:
            capas.append(layers.Dropout(tasa))
    capas += [layers.Flatten(), layers.Dense(128, activation='relu')]
    if dropout:
        capas.append(layers.Dropout(0.5))
    capas.append(layers.Dense(len(CLASES), activation='softmax'))
    return keras.Sequential(capas, name='CNN_base')

base = crear_cnn_simple(28)
base.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
base.summary()
hist_base = base.fit(X_tr28, y_tr, validation_data=(X_va28, y_va), epochs=10, batch_size=32, verbose=2)

## 7. Modelo mejorado: ResNet pequeña con Dropout (64×64)
En un bloque residual (skip connection) la entrada se suma a la salida, `y = F(x) + x`. Así el gradiente llega directo a las capas anteriores, lo que permite entrenar redes más profundas sin degradación y converger más rápido.

```
x ──► Conv3x3 ► BN ► ReLU ► Conv3x3 ► BN ──(+)──► ReLU ──►
 └───────────── atajo (Conv1x1 si cambia tamaño) ─┘
```

El *data augmentation* (solo en entrenamiento) aplica rotación de ±14°, zoom de ±10 % y traslación de ±8 %, para simular cómo varía el dibujo de cada persona.

In [ ]:
def crear_aumentacion():
    return keras.Sequential([
        layers.RandomRotation(0.04, fill_mode='constant', fill_value=0.0),
        layers.RandomZoom(0.10, fill_mode='constant', fill_value=0.0),
        layers.RandomTranslation(0.08, 0.08, fill_mode='constant', fill_value=0.0),
    ], name='data_augmentation')

def bloque_residual(x, filtros, stride=1, skip=True):
    atajo = x
    x = layers.Conv2D(filtros, 3, strides=stride, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Conv2D(filtros, 3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    if skip:
        if stride != 1 or atajo.shape[-1] != filtros:
            atajo = layers.Conv2D(filtros, 1, strides=stride, use_bias=False)(atajo)
            atajo = layers.BatchNormalization()(atajo)
        x = layers.Add()([x, atajo])
    return layers.ReLU()(x)

def crear_resnet(size=SIZE, n_clases=len(CLASES), skip=True):
    entrada = layers.Input((size, size, 1))
    x = crear_aumentacion()(entrada)
    x = layers.Conv2D(32, 3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x); x = layers.ReLU()(x)
    x = bloque_residual(x, 32, skip=skip)              # 64x64
    x = bloque_residual(x, 64, stride=2, skip=skip)    # 32x32
    x = layers.Dropout(0.2)(x)
    x = bloque_residual(x, 128, stride=2, skip=skip)   # 16x16
    x = layers.Dropout(0.3)(x)
    x = bloque_residual(x, 256, stride=2, skip=skip)   # 8x8
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    salida = layers.Dense(n_clases, activation='softmax')(x)
    return keras.Model(entrada, salida, name='DrawCAPTCHA_ResNet')

modelo = crear_resnet()
modelo.compile(optimizer=keras.optimizers.Adam(1e-3),
               loss='sparse_categorical_crossentropy', metrics=['accuracy'])
modelo.summary()

In [ ]:
def callbacks_comunes():
    return [
        keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=15, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-5, verbose=1),
    ]

callbacks = callbacks_comunes() + [
    keras.callbacks.ModelCheckpoint('drawcaptcha_resnet.keras', monitor='val_accuracy', save_best_only=True),
]
hist = modelo.fit(X_tr, y_tr, validation_data=(X_va, y_va),
                  epochs=EPOCAS, batch_size=32, callbacks=callbacks, verbose=2)

## 8. Curvas de entrenamiento

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for h, nombre, estilo in [(hist_base, 'Base 28x28', '--'), (hist, 'ResNet 64x64', '-')]:
    ax[0].plot(h.history['accuracy'], estilo, label=f'{nombre} · train')
    ax[0].plot(h.history['val_accuracy'], estilo, label=f'{nombre} · val')
    ax[1].plot(h.history['loss'], estilo, label=f'{nombre} · train')
    ax[1].plot(h.history['val_loss'], estilo, label=f'{nombre} · val')
ax[0].set_title('Exactitud (accuracy)'); ax[1].set_title('Pérdida (loss)')
for a in ax: a.set_xlabel('Época'); a.legend(fontsize=8); a.grid(alpha=.3)
plt.tight_layout(); plt.show()

## 9. Evaluación en el conjunto de prueba y matriz de confusión

In [ ]:
pred_base = base.predict(X_te28, verbose=0).argmax(1)
prob_res = modelo.predict(X_te, verbose=0)
pred_res = prob_res.argmax(1)

def matriz(y_true, y_pred, titulo):
    cm = confusion_matrix(y_true, y_pred)
    cmn = confusion_matrix(y_true, y_pred, normalize='true')
    fig, ax = plt.subplots(1, 2, figsize=(13, 5))
    ConfusionMatrixDisplay(cm, display_labels=CLASES).plot(ax=ax[0], cmap='Blues', colorbar=False)
    ConfusionMatrixDisplay(cmn, display_labels=CLASES).plot(ax=ax[1], cmap='Blues', colorbar=False, values_format='.0%')
    ax[0].set_title(f'{titulo} · conteos'); ax[1].set_title(f'{titulo} · normalizada por fila')
    plt.tight_layout(); plt.show()

matriz(y_te, pred_base, 'CNN base 28x28')
matriz(y_te, pred_res, 'ResNet mejorada 64x64')
print(classification_report(y_te, pred_res, target_names=CLASES, digits=3))

### 9.1 Comparación base vs. mejorada

In [ ]:
tabla = pd.DataFrame([
    {'Modelo': 'CNN base', 'Entrada': '28x28', 'Épocas': len(hist_base.history['loss']),
     'Parámetros': base.count_params(), 'Dropout': 'No', 'Skip connections': 'No',
     'Accuracy prueba': accuracy_score(y_te, pred_base)},
    {'Modelo': 'ResNet mejorada', 'Entrada': f'{SIZE}x{SIZE}', 'Épocas': len(hist.history['loss']),
     'Parámetros': modelo.count_params(), 'Dropout': 'Sí', 'Skip connections': 'Sí',
     'Accuracy prueba': accuracy_score(y_te, pred_res)},
])
tabla.style.format({'Accuracy prueba': '{:.2%}', 'Parámetros': '{:,}'})

### 9.2 Estudio de ablación
Cada fila añade una mejora a la anterior: base → +64×64 → +Dropout → +capas → +skip connections. Todas se entrenan de cero con la misma semilla y el mismo presupuesto de épocas (`EPOCAS` con *EarlyStopping*), así que "más épocas" no varía entre filas; su efecto se ve en la sección 9.1. Por eso la fila *base* puede diferir del modelo base de la sección 6, que usa 10 épocas.

"+capas" agrupa varios cambios, como en la tabla de la sección 3: BatchNorm, más filtros, GlobalAveragePooling y *data augmentation*.

> ⚠️ Con una sola persona en prueba y una sola semilla, las diferencias pequeñas son ruido. Esa persona puede favorecer a un modelo por su estilo de dibujo. Solo interpretamos los saltos grandes que se repitan en validación y prueba.

La celda siguiente añade 5 entrenamientos a la ejecución.

In [ ]:
def probar_variante(nombre, crear, datos):
    keras.utils.set_random_seed(SEED)             # misma semilla en todas las filas
    X_a, X_b, X_c = datos
    m = crear()
    m.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    h = m.fit(X_a, y_tr, validation_data=(X_b, y_va), epochs=EPOCAS, batch_size=32,
              callbacks=callbacks_comunes(), verbose=0)
    acierto = lambda A, b: accuracy_score(b, m.predict(A, verbose=0).argmax(1))
    print(f'{nombre}: {len(h.history["loss"])} épocas')
    return {'Variante': nombre, 'Entrada': f'{X_a.shape[1]}x{X_a.shape[1]}', 'Parámetros': m.count_params(),
            'Épocas': len(h.history['loss']), 'Accuracy validación': acierto(X_b, y_va), 'Accuracy prueba': acierto(X_c, y_te)}

datos_28, datos_64 = (X_tr28, X_va28, X_te28), (X_tr, X_va, X_te)
ablacion = pd.DataFrame([
    probar_variante('base', lambda: crear_cnn_simple(28), datos_28),
    probar_variante('+64×64', lambda: crear_cnn_simple(SIZE), datos_64),
    probar_variante('+Dropout', lambda: crear_cnn_simple(SIZE, dropout=True), datos_64),
    probar_variante('+capas', lambda: crear_resnet(skip=False), datos_64),
    probar_variante('+skip connections', lambda: crear_resnet(), datos_64),
])

personas_prueba = len(set(autores[idx_te]))
if personas_prueba == 1:
    print('⚠️ SOLO 1 PERSONA EN PRUEBA: las diferencias pequeñas entre filas son ruido, no mejoras reales.')
print(f'⚠️ Una sola semilla y {len(y_te)} imágenes de prueba de {personas_prueba} persona(s): '
      f'cada imagen mueve el accuracy {100 / len(y_te):.2f} puntos.')
ablacion.style.format({'Accuracy validación': '{:.2%}', 'Accuracy prueba': '{:.2%}', 'Parámetros': '{:,}'})

### 9.3 ¿En qué se equivoca la red?

In [ ]:
errores = np.where(pred_res != y_te)[0]
print(f'{len(errores)} errores de {len(y_te)} imágenes de prueba')
n = min(12, len(errores))
if n:
    fig, ejes = plt.subplots(1, n, figsize=(1.6 * n, 2.3), squeeze=False)
    for a, k in zip(ejes[0], errores[:n]):
        a.imshow(X_te[k, ..., 0], cmap='gray'); a.axis('off')
        a.set_title(f'real: {CLASES[y_te[k]]}\npred: {CLASES[pred_res[k]]}\n{prob_res[k].max():.0%}', fontsize=8)
    plt.tight_layout(); plt.show()

### 9.4 Robustez ante transformaciones
Evaluamos los dos modelos ya entrenados con el conjunto de prueba rotado, reducido y desplazado. La primera fila es el conjunto sin transformar. Nada se entrena ni se ajusta con estas imágenes. Las demás filas superan los rangos del *data augmentation* de la sección 7.

Las transformaciones se aplican después de `preprocesar()`. En el CAPTCHA esa función recorta y centra cada dibujo, así que un dibujo pequeño o movido llega a la red ya normalizado. Los escenarios de escala y traslación miden la tolerancia del modelo y no lo que vería con un usuario real. La rotación sí llega a la red, porque `preprocesar()` no la corrige.

> ⚠️ La base no usa aumento de datos y la ResNet sí, así que la diferencia entre ambas mezcla el efecto del aumento con el de la arquitectura, y esta tabla no permite separarlos. Valen también las cautelas de la sección 9.2 sobre una sola semilla y pocas personas en prueba.

In [ ]:
def perturbar(A, angulo=0, escala=1.0, dx=0, dy=0):
    salida = []
    for img in A:
        im = Image.fromarray(np.round(img[..., 0] * 255).astype(np.uint8))
        im = im.rotate(angulo, resample=Image.BILINEAR)
        lado = max(8, int(SIZE * escala))
        im = im.resize((lado, lado), Image.LANCZOS)
        lienzo = Image.new('L', (SIZE, SIZE), 0)
        lienzo.paste(im, ((SIZE - lado) // 2 + dx, (SIZE - lado) // 2 + dy))
        salida.append(np.asarray(lienzo, dtype=np.float32)[..., None] / 255.0)
    return np.array(salida)

assert np.array_equal(perturbar(X_te), X_te), 'sin parámetros, perturbar debe devolver la misma imagen'

escenarios = {
    'Original':              dict(),
    'Rotado +20°':           dict(angulo=20),
    'Rotado -20°':           dict(angulo=-20),
    'Rotado +30°':           dict(angulo=30),
    'Reducido al 65 %':      dict(escala=0.65),
    'Reducido y movido':     dict(escala=0.65, dx=9, dy=-9),
    'Rotado 15° + reducido': dict(angulo=15, escala=0.75, dx=-5, dy=5),
}
filas = []
for nombre, kw in escenarios.items():
    X_p = perturbar(X_te, **kw)
    filas.append({'Escenario': nombre,
                  'CNN base': accuracy_score(y_te, base.predict(a28(X_p), verbose=0).argmax(1)),
                  'ResNet mejorada': accuracy_score(y_te, modelo.predict(X_p, verbose=0).argmax(1))})
robustez = pd.DataFrame(filas).set_index('Escenario')

ax = robustez.plot.barh(figsize=(9, 4.5), color=['#9aa5bd', '#2f6fed'], xlim=(0, 1))
ax.invert_yaxis(); ax.set_xlabel('Accuracy en prueba'); ax.set_title('Robustez ante transformaciones del dibujo')
for barras in ax.containers: ax.bar_label(barras, fmt='%.2f', fontsize=8, padding=2)
plt.tight_layout(); plt.show()

fig, ejes = plt.subplots(1, len(escenarios), figsize=(1.7 * len(escenarios), 2))
for a, (nombre, kw) in zip(ejes, escenarios.items()):
    a.imshow(perturbar(X_te[:1], **kw)[0, ..., 0], cmap='gray'); a.set_title(nombre, fontsize=7); a.axis('off')
plt.tight_layout(); plt.show()
print('Promedio en escenarios transformados:')
print(robustez.iloc[1:].mean().map('{:.2%}'.format).to_string())
robustez.style.format('{:.2%}')

## 10. DrawCAPTCHA: flujo de acceso
Regla de decisión: se pide la clase *c* y el acceso se concede si la red predice *c* y su probabilidad es ≥ `UMBRAL`.

* FRR (falso rechazo): un humano dibuja bien y es rechazado, lo que molesta a quien usa el sistema.
* FAR (falsa aceptación): se dibuja otra figura (o un garabato de bot) y se acepta, lo que es un riesgo de seguridad.

Simulamos intentos *genuinos* (se pide la figura que realmente se dibujó) e *impostores* (se pide una figura distinta).

Para elegir el umbral usamos solo validación, y así no ajustamos nada con la prueba. `UMBRAL` es el valor más bajo con FAR ≤ `FAR_MAX` (1 %): prioriza la seguridad y, dentro de ella, rechaza lo menos posible a los humanos. Si ningún umbral lo cumple, se toma el de FAR más bajo (en empate, el de menor FRR) y se imprime un aviso. FAR y FRR finales se miden en prueba.

In [ ]:
def acceso(prob, clase_pedida, umbral):
    return prob.argmax() == clase_pedida and prob[clase_pedida] >= umbral

def tasas(prob, y_real, umbrales):
    """FAR y FRR por umbral: genuinos piden la clase real; impostores, cada una de las otras."""
    FAR, FRR = [], []
    for u in umbrales:
        genuinos = [acceso(p, t, u) for p, t in zip(prob, y_real)]
        impostores = [acceso(p, c, u) for p, t in zip(prob, y_real) for c in range(len(CLASES)) if c != t]
        FRR.append(1 - np.mean(genuinos)); FAR.append(np.mean(impostores))
    return np.array(FAR), np.array(FRR)

umbrales = np.linspace(0.2, 0.99, 80)
prob_va = modelo.predict(X_va, verbose=0)
FAR_va, FRR_va = tasas(prob_va, y_va, umbrales)
cumplen = np.where(FAR_va <= FAR_MAX)[0]
if len(cumplen):
    k = cumplen[0]                                # umbrales va en orden ascendente: el más bajo
else:
    k = np.lexsort((FRR_va, FAR_va))[0]           # FAR más bajo; FRR desempata
    print(f'⚠️ NINGÚN UMBRAL LLEGA A FAR ≤ {FAR_MAX:.0%} EN VALIDACIÓN: se usa el de FAR más bajo.')
UMBRAL = float(umbrales[k])

FAR, FRR = tasas(prob_res, y_te, umbrales)
plt.figure(figsize=(8, 4))
plt.plot(umbrales, FAR * 100, label='FAR · falsa aceptación (prueba)')
plt.plot(umbrales, FRR * 100, label='FRR · falso rechazo (prueba)')
plt.axvline(UMBRAL, color='gray', ls='--', label=f'umbral elegido en validación = {UMBRAL:.2f}')
plt.xlabel('Umbral de confianza'); plt.ylabel('%'); plt.title('Seguridad vs. usabilidad del CAPTCHA')
plt.legend(); plt.grid(alpha=.3); plt.show()
print(f'Validación, umbral {UMBRAL:.2f}:  FAR = {FAR_va[k]:.2%}   FRR = {FRR_va[k]:.2%}')
print(f'Prueba,     umbral {UMBRAL:.2f}:  FAR = {FAR[k]:.2%}   FRR = {FRR[k]:.2%}')

### 10.1 FAR con garabatos aleatorios
La sección anterior mide a quien dibuja otra figura. Aquí simulamos un bot que traza líneas al azar: generamos `N_GARABATOS` garabatos, cada uno pasa por `preprocesar()` y se le pide una figura al azar. La red sigue teniendo 5 clases, así que solo el umbral puede rechazarlos.

El umbral es el elegido en validación y no se ajusta con estos datos. Estos garabatos son un ataque ingenuo: este FAR no mide a un atacante que optimice el trazo para engañar a la red.

In [ ]:
def garabato(rng):
    """Entre 1 y 4 trazos al azar en el mismo lienzo de la captura (280×280, 9 px)."""
    img = Image.new('RGB', (280, 280), 'white')
    lienzo = ImageDraw.Draw(img)
    for _ in range(rng.integers(1, 5)):
        pasos = rng.normal(0, 35, (rng.integers(5, 30), 2))
        puntos = np.clip(rng.uniform(40, 240, 2) + np.cumsum(pasos, axis=0), 5, 275)
        lienzo.line([tuple(p) for p in puntos], fill='black', width=9, joint='curve')
    return img

rng_garabatos = np.random.default_rng(SEED)
X_gar = np.array([preprocesar(garabato(rng_garabatos)) for _ in range(N_GARABATOS)])
pedidas = rng_garabatos.integers(len(CLASES), size=N_GARABATOS)
prob_gar = modelo.predict(X_gar, verbose=0)
FAR_gar = np.array([np.mean([acceso(p, c, u) for p, c in zip(prob_gar, pedidas)]) for u in umbrales])

fig, ax = plt.subplots(1, 2, figsize=(13, 3.8), gridspec_kw={'width_ratios': [1, 2]})
ax[0].imshow(np.vstack([np.hstack(fila) for fila in X_gar[:16, ..., 0].reshape(4, 4, SIZE, SIZE)]), cmap='gray')
ax[0].axis('off'); ax[0].set_title('Garabatos tras preprocesar')
ax[1].plot(umbrales, FAR_gar * 100, label='FAR · garabatos aleatorios')
ax[1].plot(umbrales, FAR * 100, label='FAR · otra figura (prueba)')
ax[1].axvline(UMBRAL, color='gray', ls='--', label=f'umbral elegido en validación = {UMBRAL:.2f}')
ax[1].set_xlabel('Umbral de confianza'); ax[1].set_ylabel('%'); ax[1].set_title('Falsa aceptación según el tipo de ataque')
ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()
print(f'Garabatos, umbral {UMBRAL:.2f}:  FAR = {FAR_gar[k]:.2%}  ({round(FAR_gar[k] * N_GARABATOS)} de {N_GARABATOS} intentos aceptados)')

### 10.2 Demo interactiva (Colab)
Al ejecutar la celda aparece un recuadro para dibujar la figura pedida, y el botón *Verificar* evalúa el dibujo. Para otro intento se vuelve a ejecutar la celda. Fuera de Colab no hay recuadro: la celda simula un intento genuino y uno impostor con imágenes de prueba.

In [ ]:
CANVAS_HTML = '''
<div style="font-family:Segoe UI,sans-serif;max-width:300px">
 <div style="background:#2f6fed;color:#fff;padding:10px 12px;border-radius:10px 10px 0 0">
  <small>Verificación humana</small><div style="font-size:22px;font-weight:800">Dibuja: __CLASE__</div></div>
 <canvas id="cv" width="280" height="280" style="border:2px solid #2f6fed;touch-action:none;background:#fff;display:block"></canvas>
 <button id="lim" style="padding:8px 14px;margin-top:6px">Limpiar</button>
 <button id="ok" style="padding:8px 14px;margin-top:6px;background:#2f6fed;color:#fff;border:0;border-radius:6px">Verificar</button>
</div>
<script>
var cv=document.getElementById('cv'), ctx=cv.getContext('2d'), down=false;
function fondo(){ctx.fillStyle='#fff';ctx.fillRect(0,0,280,280);}
fondo(); ctx.lineWidth=9; ctx.lineCap='round'; ctx.lineJoin='round'; ctx.strokeStyle='#000';
function pos(e){var r=cv.getBoundingClientRect();return [e.clientX-r.left,e.clientY-r.top];}
cv.onpointerdown=function(e){down=true;var p=pos(e);ctx.beginPath();ctx.moveTo(p[0],p[1]);ctx.lineTo(p[0]+0.1,p[1]);ctx.stroke();};
cv.onpointermove=function(e){if(!down)return;var p=pos(e);ctx.lineTo(p[0],p[1]);ctx.stroke();};
window.onpointerup=function(){down=false;};
document.getElementById('lim').onclick=fondo;
var dibujo=new Promise(function(res){document.getElementById('ok').onclick=function(){res(cv.toDataURL('image/png'));};});
</script>'''

def dibujo_valido(img, minimo=35, grosor=9):
    """Misma regla que dibujoValido() en captura/app.js: sin trazo o menor a 35 px no se evalúa."""
    g = 255.0 - np.asarray(img.convert('L'), dtype=np.float32)
    ys, xs = np.where(g > 40)                     # mismo corte que preprocesar()
    if len(xs) == 0:
        return False
    return max(ys.max() - ys.min(), xs.max() - xs.min()) + 1 - grosor >= minimo   # la regla mide el recorrido, sin el grosor

assert not dibujo_valido(Image.new('RGB', (280, 280), 'white'))

def mostrar_resultado(img, prob, pedida):
    ok = acceso(prob, pedida, UMBRAL)
    fig, ax = plt.subplots(1, 2, figsize=(9, 3), gridspec_kw={'width_ratios': [1, 2]})
    ax[0].imshow(img[..., 0], cmap='gray'); ax[0].axis('off'); ax[0].set_title('Entrada a la red')
    colores = ['#17a34a' if (i == pedida and ok) else '#dc3b3b' if i == prob.argmax() else '#9aa5bd' for i in range(len(CLASES))]
    ax[1].barh(CLASES, prob, color=colores); ax[1].axvline(UMBRAL, ls='--', color='gray'); ax[1].set_xlim(0, 1)
    ax[1].set_title(('ACCESO CONCEDIDO' if ok else 'ACCESO DENEGADO') + f'  (se pidió: {CLASES[pedida]})',
                    color='#17a34a' if ok else '#dc3b3b', fontweight='bold')
    plt.tight_layout(); plt.show()
    print('✅ Eres humano, puedes pasar.' if ok else '⛔ Verificación fallida, inténtalo de nuevo.')

def captcha():
    pedida = random.randrange(len(CLASES))
    try:
        from google.colab.output import eval_js  # type: ignore
        from IPython.display import HTML, display
        display(HTML(CANVAS_HTML.replace('__CLASE__', CLASES[pedida].upper())))
        datos = eval_js('dibujo')
        dibujo = Image.open(io.BytesIO(base64.b64decode(datos.split(',')[1])))
        if not dibujo_valido(dibujo):
            print('⛔ ACCESO DENEGADO: lienzo vacío o dibujo demasiado pequeño. Vuelve a ejecutar la celda.')
            return
        img = preprocesar(dibujo)
    except ImportError:   # fuera de Colab: intento genuino e impostor con imágenes de prueba
        otra = (pedida + random.randrange(1, len(CLASES))) % len(CLASES)
        for intento, clase in [('genuino', pedida), ('impostor', otra)]:
            print(f'(Fuera de Colab) Intento {intento}: se pide {CLASES[pedida]} y se entrega un dibujo de {CLASES[clase]}')
            img = X_te[np.random.choice(np.where(y_te == clase)[0])]
            mostrar_resultado(img, modelo.predict(img[None], verbose=0)[0], pedida)
        return
    prob = modelo.predict(img[None], verbose=0)[0]
    mostrar_resultado(img, prob, pedida)

captcha()

## 11. Guardar el modelo

In [ ]:
modelo.save('drawcaptcha_resnet.keras')
print('Modelo guardado: drawcaptcha_resnet.keras')
# Para descargarlo desde Colab:
# from google.colab import files; files.download('drawcaptcha_resnet.keras')

### 11.1 Exportar las métricas
La celda guarda las métricas en `resultados.json`, que no se sube a git. El archivo solo cuenta autores y no incluye sus nombres. El campo `datos_sinteticos` marca las ejecuciones de prueba, cuyas métricas no se reportan.

In [ ]:
def brecha(h, epoca):
    return float(h.history['accuracy'][epoca] - h.history['val_accuracy'][epoca])

mejor_epoca = int(np.argmax(hist.history['val_accuracy']))   # EarlyStopping restaura los pesos de esa época
resultados = {
    'datos_sinteticos': 'sintetico' in RUTA.lower(),
    'modo_split': MODO_SPLIT,
    'n_total': len(y), 'n_train': len(y_tr), 'n_val': len(y_va), 'n_test': len(y_te),
    'n_autores': len(set(autores)), 'n_autores_prueba': len(set(autores[idx_te])),
    'acc_base': accuracy_score(y_te, pred_base), 'acc_resnet': accuracy_score(y_te, pred_res),
    'epocas_base': len(hist_base.history['loss']), 'epocas_resnet': len(hist.history['loss']),
    'params_base': base.count_params(), 'params_resnet': modelo.count_params(),
    'brecha_base': brecha(hist_base, -1), 'brecha_resnet': brecha(hist, mejor_epoca),
    'umbral': UMBRAL, 'far_max': FAR_MAX,
    'FAR_val': float(FAR_va[k]), 'FRR_val': float(FRR_va[k]),
    'FAR': float(FAR[k]), 'FRR': float(FRR[k]), 'FAR_garabatos': float(FAR_gar[k]),
    'cm_base': confusion_matrix(y_te, pred_base).tolist(), 'cm_resnet': confusion_matrix(y_te, pred_res).tolist(),
    'reporte': classification_report(y_te, pred_res, target_names=CLASES, output_dict=True),
    'ablacion': ablacion.to_dict('records'),
    'robustez': robustez.to_dict('index'),
}
with open('resultados.json', 'w', encoding='utf-8') as f:
    json.dump(resultados, f, indent=1, ensure_ascii=False)
print('Métricas guardadas en resultados.json')

## 12. Conclusiones
Pendientes de la ejecución con el dataset real.

1. La CNN base (28×28, 10 épocas) obtuvo [completar] % de exactitud; la ResNet mejorada (64×64, Dropout, skip connections, augmentation) obtuvo [completar] %.
2. Según la matriz de confusión, las clases que más se confunden son [completar] y [completar], porque [completar].
3. Con Dropout y data augmentation, la brecha entre train y val pasó de [completar] a [completar].
4. Con el umbral elegido en validación ([completar]) el CAPTCHA tiene en prueba FAR = [completar] % y FRR = [completar] %. Equilibrio entre seguridad y usabilidad: [completar].
5. Frente a garabatos aleatorios, el FAR con ese umbral fue [completar] %. Comparación con el FAR de la conclusión 4: [completar]. Un ataque ingenuo no mide a un atacante que optimice el trazo.
6. En la ablación, la mejora que más aportó fue [completar], de [completar] % a [completar] % en validación y de [completar] % a [completar] % en prueba. Solo interpretamos saltos grandes que se repitan en ambos conjuntos; con una persona en prueba, las diferencias pequeñas son ruido.
7. En los escenarios transformados de la sección 9.4, la base promedió [completar] % y la ResNet [completar] %. Esa diferencia mezcla el aumento de datos con la arquitectura, y los escenarios de escala y traslación no reflejan lo que `preprocesar()` entrega en el CAPTCHA.
8. Trabajo futuro: más dibujantes, clase "garabato/bot" para rechazar trazos aleatorios, evaluar ataques con imágenes generadas, uso en el navegador con TensorFlow.js.